In [34]:
import numpy as np
import pandas as pd

In [35]:
def generate_ordinal_customer_data(n=500,noise=0.1):
    
    rng = np.random.default_rng(seed=42) # random number generator

    # Ordered categories
    education_levels = [
            'high_school',
            'bachelor',
            'master',
            'phd'
    ]

    job_levels = [
        'entry',
        'junior',
        'mid',
        'senior',
        'lead'
    ]
    
    education = rng.choice(
        education_levels,
        size=n,
        p=[0.35, 0.40, 0.20, 0.05]
    )

    job_level = rng.choice(
        job_levels,
        size=n,
        p=[0.25, 0.25, 0.25, 0.20, 0.05]
    ) # more people at lower level jobs (few leads)

    city = rng.choice(
        ['Berlin', 'Munich', 'Geneva', 'Vienna', 'London', 'Cambridge'],
        size=n
    ) # Nominal dataa(non-ordinal) for One-hot encoding
    
    age = rng.integers(20, 60, size=n)

    income = rng.normal(
        loc=40000,
        scale=15000,
        size=n
    ).astype(int)

    income = np.clip(income, 8000, 150000)

    education_score = {
        'high_school': 0,
        'bachelor': 1,
        'master': 2,
        'phd': 3
    }

    job_score = {
        'entry': 0,
        'junior': 1,
        'mid': 2,
        'senior': 3,
        'lead': 4,
    }

    edu_numeric = np.array(
        [education_score[e] for e in education]
    )

    job_numeric = np.array(
        [job_score[j] for j in job_level]
    )

    approval_probability = (
        0.15 # base approval probability(0.2)
        + 0.12 * edu_numeric # hugher education = higher approval chance
        + 0.10 * job_numeric # higher the level, higehr the approval chance
        + 0.000005 * income # higher income = higher approval chance
        - 0.002 * (age - 30).clip(min=0) # old age = after age = 30, reduces approval chance
    )

    approval_probability = np.clip(
        approval_probability, 0, 1
    )

    approved = (
        rng.random(n) < approval_probability
    ) # True = Approved, False = Rejected

    noise_mask = rng.random(n) < noise
    approved[noise_mask] == ~approved[noise_mask]

    approved = np.where(approved, 'yes', 'no')
    
    return pd.DataFrame(
        {
            'age': age,
            'income': income,
            'education_level': education,
            'job_level': job_level,
            'city': city,
            'approved': approved,
        }
    )

In [36]:
df = generate_ordinal_customer_data(1000)
df = df.to_csv('ordinal_interview_dummy_dataset.csv', index=None)